# Data Science for Business - The lasso on Ames Housing

*Session 5 · ISLP 6.2.2 The Lasso · 6.2.3 Selecting the Tuning Parameter · 6.4 High Dimensions*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score, root_mean_squared_error

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_05/ameshousing.csv')

In [ ]:
data.head()

## Prepare data

First, we will remove some columns that are not useful for our task.

In [ ]:
data = data.drop(['YrSold', 'MoSold', 'SaleCondition', 'SaleType'], axis=1)

Next, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets.

In [ ]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Finally, we prepare the predictors for the lasso. It is important to learn the preprocessing (e.g., means and standard deviations) from the training set only, and then mechanistically repeat the same steps on the test set. So far, we did this by hand. From now on, we let scikit-learn do the bookkeeping: we describe the preprocessing once and combine it with the model in a *pipeline* (see below).

Typically, feature engineering depends strongly on the datatype of the variables. Hence, we will first determine which variables are categorical and which are numerical. Subsequently, we will transform these variables separately.

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns

The numerical variables are standardized (subtract the mean, divide by the standard deviation). This is especially important for the lasso: the penalty treats all coefficients alike, so the predictors need to be on comparable scales.

In [ ]:
num_transformer = StandardScaler()

The categorical variables must be transformed into numerical representations, e.g., by one-hot encoding them. We then standardize the dummies as well, so that the penalty treats them like the numerical predictors. Two steps in a row: we chain them with a `Pipeline`, a list of `(name, step)` pairs.

In [ ]:
cat_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ('scale', StandardScaler())
])

A `ColumnTransformer` applies each transformer to its own columns and glues the results together side by side. `set_output(transform='pandas')` makes it return a dataframe with readable column names (`num__...` for numerical, `cat__...` for one-hot encoded predictors).

In [ ]:
preprocess = ColumnTransformer([
    ('num', num_transformer, numerical_features),
    ('cat', cat_transformer, categorical_features)
]).set_output(transform='pandas')

Let's look at what the preprocessing produces. Here we fit it on the training data ourselves; later, the pipeline will do this for us.

In [ ]:
X_train_prep = preprocess.fit_transform(X_train)
X_train_prep.head()

In [ ]:
X_train_prep.shape

## The lasso

The lasso minimises $\text{RSS} + \lambda \sum_j |\beta_j|$ (ISLP 6.2.2, eq. 6.7). A larger λ shrinks the coefficients more and sets some of them exactly to zero: automatic feature selection.

sklearn calls λ `alpha` and scales the RSS by $\frac{1}{2n}$: `Lasso` minimises $\frac{1}{2n}\text{RSS} + \alpha \sum_j |\beta_j|$. Multiplying by $2n$ shows that $\alpha = \lambda / (2n)$, so sklearn's alpha values are much smaller than the book's λ values for the same model.

We start with a **strong penalty**, `alpha = 10000`, and fit the model on the training data. With such a large alpha, the lasso sets most coefficients to zero, so the model will probably be too simple.

We combine the preprocessing and the lasso in a **pipeline**: a chain of steps that behaves like a single model. `fit` runs all steps in order on the training data (fit the preprocessing, transform the data, fit the lasso); `predict` applies the same, already fitted, steps to new data. So we can pass the raw `X_train` and `X_test` and never have to transform the test set by hand.

Pipelines really pay off in cross-validation: the whole pipeline is refitted in every fold, so the preprocessing is learned from the training folds only and nothing leaks from the held-out fold.

Each step has a name (here `'preprocess'` and `'model'`). We reach a step via `named_steps['<name>']`, and set or tune a parameter of a step with the `<step>__<parameter>` syntax (two underscores), e.g., `model__alpha`.

In [ ]:
lasso_mod = Pipeline([
    ('preprocess', preprocess),
    ('model', Lasso(alpha=10000))
])
lasso_mod.fit(X_train, y_train)

Evaluate the model on both training and test set using *R2* and *RMSE* as metrics.

In [ ]:
# Training data
pred_train = lasso_mod.predict(X_train)
r2_train = r2_score(y_train, pred_train)
rmse_train = root_mean_squared_error(y_train, pred_train)
print('R2 on training set:', round(r2_train, 2))
print('RMSE on training set:', round(rmse_train, 2))

print("===")

# Test data
pred_test = lasso_mod.predict(X_test)
r2_test = r2_score(y_test, pred_test)
rmse_test = root_mean_squared_error(y_test, pred_test)
print('R2 on test set:', round(r2_test, 2))
print('RMSE on test set:', round(rmse_test, 2))

How many coefficients are not zero?

In [ ]:
coefs = lasso_mod.named_steps['model'].coef_
print((coefs != 0).sum(), 'of', len(coefs), 'features kept')

Only a handful of features survive, and the RMSE is high on both the training and the test set: with this much shrinkage, the model **underfits**. Which alpha would be better? Let's first look at what happens to the coefficients across many alpha values.

### The coefficient path

Next, we try 100 alpha values between $10^2$ and $10^5$, evenly spaced on a log scale (like the λ grids in ISLP). The loop below sets the lasso's alpha via `set_params(model__alpha=...)`, refits the pipeline, and collects the estimated coefficients (with the feature names from the fitted preprocessing step) in a dataframe.

In [ ]:
alphas = np.logspace(2, 5, 100)

lasso_mod = Pipeline([
    ('preprocess', preprocess),
    ('model', Lasso())
])

results = []
for a in alphas:
    result = {}
    lasso_mod.set_params(model__alpha=a)
    lasso_mod.fit(X_train, y_train)

    result["alpha"] = a
    feature_names = lasso_mod.named_steps['preprocess'].get_feature_names_out()
    for name, value in zip(feature_names, lasso_mod.named_steps['model'].coef_):
        result[name] = value

    results.append(result)

In [ ]:
results_df = pd.DataFrame(results)
results_df.head()

Let's visualize how the coefficients shrink with increasing alpha. First, we wrangle the data into long format (one row per alpha and feature).

In [ ]:
results_df_long = results_df.melt(id_vars=['alpha'], var_name='feature', value_name='coefficient')
results_df_long.head()

In [ ]:
sns.lineplot(data=results_df_long, x='alpha', y='coefficient', hue='feature', legend=False)
plt.xscale('log')
plt.xlabel('alpha (log scale)')
plt.ylabel('Standardized coefficients')
plt.show()

Each line is one coefficient. Moving right, they shrink towards zero and drop out one by one; the survivors are the strongest predictors (cf. ISLP Fig. 6.6).

## Hyperparameter tuning

The proper way to choose alpha is k-fold cross-validation on the **training** set (ISLP 6.2.3). `GridSearchCV` takes our pipeline and a grid of candidate values: for every alpha and every fold, it fits the *whole pipeline* (preprocessing and lasso) on the training folds and computes the RMSE on the held-out fold. So the scaler never sees the held-out fold, and nothing leaks. Finally, it picks the alpha with the lowest average CV error. We shuffle the folds with a fixed seed so that the result is reproducible.

The grid is given as a dictionary. Its keys use the `<step>__<parameter>` syntax: `model__alpha` means "the parameter `alpha` of the step named `'model'`" (step name, two underscores, parameter name). We use a coarser grid of 20 alphas than for the coefficient path, because each alpha now means 5 full pipeline fits. `scoring='neg_root_mean_squared_error'` tells it to use the RMSE (negated, because sklearn always maximises scores), and `n_jobs=-1` uses all CPU cores.

In [ ]:
alphas_cv = np.logspace(2, 5, 20)

lasso_mod_cv = GridSearchCV(
    Pipeline([
        ('preprocess', preprocess),
        ('model', Lasso())
    ]),
    param_grid={'model__alpha': alphas_cv},
    cv=KFold(5, shuffle=True, random_state=42),
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
lasso_mod_cv.fit(X_train, y_train)

Which alpha value leads to the best out-of-sample predictive accuracy?

In [ ]:
lasso_mod_cv.best_params_['model__alpha']

Plot the CV RMSE against alpha (cf. ISLP Fig. 6.13). `cv_results_` holds, among other things, the mean score over the 5 folds for each alpha (`mean_test_score`); we flip its sign to get the RMSE. The dashed line marks the chosen alpha.

In [ ]:
cv_rmse = -lasso_mod_cv.cv_results_['mean_test_score']

plt.plot(alphas_cv, cv_rmse)
plt.axvline(lasso_mod_cv.best_params_['model__alpha'], color='black', ls='--')
plt.xscale('log')
plt.xlabel('alpha (log scale)')
plt.ylabel('CV RMSE')
plt.show()

Refit the model with the best alpha value on the full training data (`GridSearchCV` already does this internally and stores the result in `best_estimator_`; we do it explicitly here).

In [ ]:
lasso_mod_tuned = Pipeline([
    ('preprocess', preprocess),
    ('model', Lasso(alpha=lasso_mod_cv.best_params_['model__alpha']))
])
lasso_mod_tuned.fit(X_train, y_train)

How many features survived?

In [ ]:
coefs = lasso_mod_tuned.named_steps['model'].coef_
print((coefs != 0).sum(), 'of', len(coefs), 'features kept')

Evaluate this model on the test set.

In [ ]:
# Training data
pred_train = lasso_mod_tuned.predict(X_train)
r2_train = r2_score(y_train, pred_train)
rmse_train = root_mean_squared_error(y_train, pred_train)
print('R2 on training set:', round(r2_train, 2))
print('RMSE on training set:', round(rmse_train, 2))

print("===")

# Test data
pred_test = lasso_mod_tuned.predict(X_test)
r2_test = r2_score(y_test, pred_test)
rmse_test = root_mean_squared_error(y_test, pred_test)
print('R2 on test set:', round(r2_test, 2))
print('RMSE on test set:', round(rmse_test, 2))

Compare this with our first model (`alpha = 10000`): cross-validation found a much smaller penalty, and the test RMSE is clearly lower. At the same time, the tuned model uses only a fraction of the ~290 features: the lasso has selected the ones that matter. (The CV curve is fairly flat around its minimum, so somewhat smaller alphas with many more features would predict about equally well; the lasso lets us choose the simpler model.)

## Your turn!

**Your turn!** Ridge regression (ISLP 6.2.1) shrinks coefficients towards zero but never sets them exactly to zero.

1. Build a pipeline with our `preprocess` step and a `RidgeCV` model (from `sklearn.linear_model`) with `alphas=np.logspace(-2, 4, 100)` and 5-fold CV, and fit it on `X_train`. Note: sklearn's `Ridge` minimises $\text{RSS} + \alpha \sum_j \beta_j^2$, so here alpha *is* the book's λ.
2. Compare its test RMSE and the number of non-zero coefficients with the tuned lasso model. Which model would you prefer, and why?

In [ ]:
# YOUR CODE HERE